In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

BASE_URL = "https://www.rekrute.com"
SEARCH_URL = "https://www.rekrute.com/offres.html"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

search_terms = [

    # ── Data & Analytics ──────────────────────────────
    "Data Engineer",
    "Data Analyst",
    "Data Scientist",
    "Business Intelligence",
    "Power BI",
    "Tableau",
    "ETL Developer",
    "Data Warehouse",
    "Big Data",
    "Ingénieur Data",
    "Analyste Données",
    "Data Architect",
    "Data Governance",
    "Reporting Analyst",
    "dbt Developer",

    # ── Cloud & DevOps ────────────────────────────────
    "DevOps Engineer",
    "Cloud Engineer",
    "Azure Engineer",
    "AWS Engineer",
    "Ingénieur Cloud",
    "Ingénieur DevOps",
    "Site Reliability Engineer",
    "Kubernetes",
    "Terraform",
    "CI/CD Engineer",
    "DevSecOps",
    "Platform Engineer",

    # ── AI & Machine Learning ─────────────────────────
    "Machine Learning",
    "Intelligence Artificielle",
    "AI Engineer",
    "NLP Engineer",
    "MLOps",
    "Deep Learning",
    "Computer Vision",
    "Generative AI",
    "LLM Engineer",

    # ── Software Engineering ──────────────────────────
    "Développeur Full Stack",
    "Développeur Backend",
    "Développeur Frontend",
    "Software Engineer",
    "Ingénieur Logiciel",
    "Java Developer",
    "Développeur Java",
    "Python Developer",
    "Développeur Python",
    "Développeur .NET",
    ".NET Developer",
    "Node.js Developer",
    "Développeur Node.js",
    "React Developer",
    "Développeur React",
    "Angular Developer",
    "Développeur Angular",
    "Microservices",
    "API Developer",

    # ── Web & PHP ─────────────────────────────────────
    "Développeur PHP",
    "PHP Developer",
    "Laravel Developer",
    "Symfony Developer",
    "Développeur Web",
    "Web Developer",
    "WordPress Developer",
    "Vue.js Developer",
    "Frontend Developer",
    "Développeur Frontend",

    # ── Mobile ────────────────────────────────────────
    "Développeur Mobile",
    "Mobile Developer",
    "Android Developer",
    "iOS Developer",
    "Flutter Developer",
    "React Native Developer",
    "Kotlin Developer",
    "Swift Developer",

    # ── Architecture & Leadership ─────────────────────
    "Architecte Logiciel",
    "Solution Architect",
    "Enterprise Architect",
    "Technical Lead",
    "Lead Développeur",
    "Engineering Manager",
    "CTO",
    "DSI",
    "Directeur Informatique",

    # ── Cybersecurity ─────────────────────────────────
    "Ingénieur Cybersécurité",
    "Cybersecurity Engineer",
    "Security Analyst",
    "Analyste Sécurité",
    "Responsable Sécurité SI",
    "SOC Analyst",
    "Penetration Tester",
    "Cloud Security",
    "SIEM Engineer",
    "ISO 27001",
    "GRC Analyst",
    "RSSI",

    # ── Infrastructure & Networking ───────────────────
    "Ingénieur Réseaux",
    "Network Engineer",
    "Ingénieur Systèmes",
    "Systems Administrator",
    "Administrateur Systèmes",
    "Linux Administrator",
    "Database Administrator",
    "Administrateur Base de Données",
    "Ingénieur Infrastructure",

    # ── Telecom ───────────────────────────────────────
    "Ingénieur Télécoms",
    "Telecom Engineer",
    "VoIP Engineer",
    "RF Engineer",
    "NOC Engineer",
    "IP Network Engineer",

    # ── ERP & Business Systems ────────────────────────
    "Consultant SAP",
    "SAP Consultant",
    "SAP ABAP",
    "Oracle ERP",
    "Salesforce Developer",
    "Développeur Salesforce",
    "Dynamics 365",
    "ServiceNow Developer",
    "ERP Consultant",
    "Consultant Fonctionnel",
    "Consultant Technique",
    "AMOA",
    "MOA Consultant",
    "Ingénieur d'Études",

    # ── Microsoft Ecosystem ───────────────────────────
    "Power Platform",
    "Power Apps",
    "Power Automate",
    "SharePoint Developer",
    "Azure Administrator",
    "Microsoft 365",

    # ── QA & Testing ──────────────────────────────────
    "Ingénieur QA",
    "QA Engineer",
    "Test Automation",
    "Testeur Logiciel",
    "Performance Engineer",
    "SDET",

    # ── IT Support & Helpdesk ─────────────────────────
    "Support Informatique",
    "Technicien Helpdesk",
    "Technicien Informatique",
    "IT Support",
    "IT Technician",
    "Field Service Engineer",

    # ── Product & Project ─────────────────────────────
    "Chef de Projet IT",
    "Project Manager IT",
    "Product Manager",
    "Product Owner",
    "Scrum Master",
    "Agile Coach",
    "Business Analyst",
    "Analyste Fonctionnel",

    # ── Embedded & Industrial ─────────────────────────
    "Ingénieur Systèmes Embarqués",
    "Embedded Systems Engineer",
    "Firmware Engineer",
    "SCADA Engineer",
    "Ingénieur Automatisme",
    "Automation Engineer",
    "IoT Engineer",
    "C++ Developer",

    # ── IT Consulting & Offshoring ────────────────────
    "IT Consultant",
    "Ingénieur de Développement",
    "Consultant IT",
    "Architecte Cloud",
    "Architecte Solutions",

    # ── Digital Marketing & E-commerce ───────────────
    "SEO Specialist",
    "Digital Marketing",
    "E-commerce Developer",
    "CRM Developer",
    "Marketing Automation",

    # ── GIS & Geospatial ──────────────────────────────
    "GIS Developer",
    "Géomaticien",
    "GIS Analyst",

    # ── Blockchain & Fintech ──────────────────────────
    "Blockchain Developer",
    "Fintech Developer",
    "Smart Contract",

    # ── IT Governance & Risk ──────────────────────────
    "IT Auditor",
    "Auditeur Informatique",
    "DPO",
    "Risk Manager IT",
    "CISO",

    # ── French Generic Terms (high volume on Rekrute) ─
    "Ingénieur Informatique",
    "Développeur",
    "Ingénieur Développement",
    "Responsable Informatique",
    "Technicien Réseaux",
    "Administrateur Réseaux",
]


def get_job_detail(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")
        desc_div = soup.select_one(".job-detail") or soup.select_one(".detailsJob")
        return desc_div.get_text(separator=" ", strip=True) if desc_div else ""
    except:
        return ""


def scrape_rekrute(search_term="", max_pages=20):
    jobs = []
    page = 1

    while page <= max_pages:
        print(f"  Page {page} — '{search_term}'...")
        params = {"p": page, "s": 1, "q": search_term}

        try:
            res = requests.get(SEARCH_URL, params=params, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(res.text, "html.parser")
            job_cards = soup.select(".post-id")

            if not job_cards:
                break

            for card in job_cards:
                try:
                    title_tag    = card.select_one("a.titreJob") or card.select_one("h2 a")
                    company_tag  = card.select_one(".recruteur") or card.select_one(".company")
                    location_tag = card.select_one(".location")
                    date_tag     = card.select_one(".date")
                    contract_tag = card.select_one(".contrat")
                    exp_tag      = card.select_one(".experience")
                    sector_tag   = card.select_one(".secteur")
                    salary_tag   = card.select_one(".salary") or card.select_one(".salaire")

                    title    = title_tag.get_text(strip=True)    if title_tag    else ""
                    job_url  = BASE_URL + title_tag["href"]      if title_tag    else ""
                    company  = company_tag.get_text(strip=True)  if company_tag  else ""
                    location = location_tag.get_text(strip=True) if location_tag else ""
                    date     = date_tag.get_text(strip=True)     if date_tag     else ""
                    contract = contract_tag.get_text(strip=True) if contract_tag else ""
                    exp      = exp_tag.get_text(strip=True)      if exp_tag      else ""
                    sector   = sector_tag.get_text(strip=True)   if sector_tag   else ""
                    salary   = salary_tag.get_text(strip=True)   if salary_tag   else ""

                    jobs.append({
                        "source":      "rekrute",
                        "title":       title,
                        "company":     company,
                        "location":    location,
                        "date_posted": date,
                        "job_type":    contract,
                        "experience":  exp,
                        "sector":      sector,
                        "salary":      salary,
                        "job_url":     job_url,
                        "search_term": search_term,
                        "scraped_at":  datetime.now().isoformat(),
                    })
                except Exception as e:
                    print(f"    → Card error: {e}")
                    continue

            print(f"    → {len(job_cards)} jobs found")
            page += 1
            time.sleep(1.5)

        except Exception as e:
            print(f"  → Page failed: {e}")
            break

    return jobs


# ── Run ───────────────────────────────────────────────────────────────────────
all_jobs = []

for term in search_terms:
    results = scrape_rekrute(search_term=term, max_pages=10)
    all_jobs.extend(results)
    print(f"  → Cumulative: {len(all_jobs)} jobs\n")

df = pd.DataFrame(all_jobs)

# Deduplicate
before = len(df)
df = df.drop_duplicates(subset=["job_url"])
print(f"Deduplication: {before} → {len(df)} unique jobs")

# Clean
def clean_text(val):
    if isinstance(val, str):
        val = val.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
        val = ' '.join(val.split())
    return val

df = df.apply(lambda col: col.map(clean_text))

df.to_csv(
    "../data/jobs_rekrute.tsv",
    sep="\t",
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)

print(f"Saved {len(df)} jobs to jobs_rekrute.tsv")

  Page 1 — 'Data Engineer'...
    → 10 jobs found
  Page 2 — 'Data Engineer'...
    → 10 jobs found
  Page 3 — 'Data Engineer'...
    → 10 jobs found
  Page 4 — 'Data Engineer'...
    → 10 jobs found
  Page 5 — 'Data Engineer'...
    → 10 jobs found
  Page 6 — 'Data Engineer'...
    → 10 jobs found
  Page 7 — 'Data Engineer'...
    → 10 jobs found
  Page 8 — 'Data Engineer'...
    → 10 jobs found
  Page 9 — 'Data Engineer'...
    → 10 jobs found
  Page 10 — 'Data Engineer'...
    → 10 jobs found
  → Cumulative: 100 jobs

  Page 1 — 'Data Analyst'...
    → 10 jobs found
  Page 2 — 'Data Analyst'...
    → 10 jobs found
  Page 3 — 'Data Analyst'...
    → 10 jobs found
  Page 4 — 'Data Analyst'...
    → 10 jobs found
  Page 5 — 'Data Analyst'...
    → 10 jobs found
  Page 6 — 'Data Analyst'...
    → 10 jobs found
  Page 7 — 'Data Analyst'...
    → 10 jobs found
  Page 8 — 'Data Analyst'...
    → 10 jobs found
  Page 9 — 'Data Analyst'...
    → 10 jobs found
  Page 10 — 'Data Analyst'...
  

In [2]:
df=df.drop(columns=["scraped_at"])

In [3]:
df.to_csv(
    "../data/jobs_rekrute.tsv",
    sep="\t",
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)